In [ ]:
print("Hello World!")

In [ ]:
import findspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession

spark1 = SparkSession.builder.appName("PracticeApp").master("local[*]").getOrCreate()

print("Spark Session for Practice is Created")
print("Spark version: ", spark1.version)
print("------------- Happy Codding ----------")

In [ ]:
data = [
    {"name":"masud rana", "cgpa":3.60, "address":"Kanchan, Rupganj, Narayanganj", "university":"Green University of Bangladesh"}
]

df = spark1.createDataFrame(data)
df.show(truncate=False)


## Dwonload 'csv' data from kaggle

In [ ]:
import kagglehub
import shutil
from pathlib import Path

downloaded_dir = kagglehub.dataset_download("karkavelrajaj/amazon-sales-dataset", force_download=True)
source_dir = Path(downloaded_dir)

if not source_dir.exists() and not source_dir.is_dir():
    raise FileNotFoundError("Maybe kaggle unable to download the dataset!")

files = [f for f in source_dir.iterdir() if f.is_file()]

if len(files) != 1:
    raise RuntimeError("It should contain exactly one file!")

source_file = files[0]
source_file_ext = source_file.suffix
print("source file path and extentions: ", source_file, source_file_ext)


project_root = Path.cwd()
target_dir = project_root / "data" / "raw"
target_dir.mkdir(parents=True, exist_ok=True)
print("project root: ", project_root, "target dir: ", target_dir)

target_file_name = f"amazon_sales_raw{source_file_ext}"
target_file_path = target_dir / target_file_name
print("target file name: ", target_file_name, "target file path: ", target_file_path)
shutil.move(str(source_file), str(target_file_path)) 

### Read csv file

In [ ]:
# print schema data types
from pathlib import Path
target_file_path = Path.cwd() / "data" / "raw"
df = (
    spark1.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv(str(target_file_path))
)
df.printSchema()

In [ ]:
df.count()

In [ ]:
# print all columns
print(len(df.columns))

df.columns

In [ ]:
df.show(10)

In [ ]:
# count of non-null values in each column
from pyspark.sql.functions import col, count

df.select([count(c).alias(c) for c in df.columns]).show()

In [ ]:
# count null values in each column
from pyspark.sql.functions import col, sum, when

df.select(
    [sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns]
).show()

In [ ]:
# rename the product_name to product_title
df = df.withColumnRenamed("product_name", "product_title")

In [ ]:
df.printSchema()

## verify every column is correct

In [ ]:
# check other values interefare with discounted_price column
from pyspark.sql.functions import col

invalid_discounted_price_df = df.filter(
    col("discounted_price").isNotNull() &
    ~col("discounted_price").rlike(r"^₹[\d,]+(\.\d+)?$")
)

invalid_discounted_price_df.select("discounted_price").show(truncate=False)


In [ ]:
# remove special characters and cast  
from pyspark.sql.functions import when, regexp_replace, col

def remove_symbol_and_cast(column_name, target_type):
    cleaned_symbol = regexp_replace(col(column_name), "[^\d.]", "")
    return when(
        cleaned_symbol.rlike(r"^\d+(\.\d+)?$"),
        cleaned_symbol.try_cast(target_type)
    ).otherwise(None)


df = df.withColumn("discounted_price", remove_symbol_and_cast("discounted_price", "float")) \
    .withColumn("actual_price", remove_symbol_and_cast("actual_price", "float")) \
    .withColumn("discount_percentage", remove_symbol_and_cast("discount_percentage", "int")) \
    .withColumn("rating_count", remove_symbol_and_cast("rating_count", "int"))

In [ ]:
df.show(5)

In [ ]:
df.printSchema()

In [ ]:
df = df.withColumn("rating", col("rating").try_cast("float"))

In [ ]:
df.printSchema()

In [ ]:
# see null values in each columns
df.select(
    [sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns]
).show()

In [ ]:
# drop rows will null values
df = df.dropna(subset=[c for c in df.columns])

df.count()

In [ ]:
# see null values in each columns
df.select(
    [sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns]
).show()

In [ ]:
df.count()

In [ ]:
df.show(10)

In [ ]:
# count and sum of numeric columns
from pyspark.sql.functions import col, count, sum, round as spark_round
from pyspark.sql.types import IntegerType, FloatType, DoubleType, LongType, ShortType

# filter only numeric columns
numeric_types = (IntegerType, FloatType, DoubleType, LongType, ShortType)
numeric_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, numeric_types)
]

numeric_columns

In [ ]:
# count of numeric columns
df.select([count(c).alias(c) for c in numeric_columns]).show()

# sum of each numeric columns
df.select([spark_round(sum(col(c)), 3).alias(c) for c in numeric_columns]).show()

In [ ]:
df.show(5)

## Transformation apply

In [ ]:
# add price difference column
df = df.withColumn("price_difference", col("actual_price") - col("discounted_price"))

df.show(5)

In [ ]:
# rating bucket
df = df.withColumn(
    "rating_bucket",
    when(col("rating") >= 4.5, "Excellent")
    .when(col("rating") >= 4.0, "Good")
    .when(col("rating") >= 3.0, "Average")
    .otherwise("Poor")
)

df.show(10)

In [ ]:
# review strength bucket
df = df.withColumn(
    "rating_strength",
    when(col("rating_count") >= 10000, "Very High")
    .when(col("rating_count") >= 1000, "High")
    .when(col("rating_count") >= 100, "Medium")
    .otherwise("Low")
)

In [ ]:
df.show(10)

In [ ]:
# discount percentage bucket
from pyspark.sql.functions import when

df = df.withColumn(
    "discount_bucket",
    when(col("discount_percentage") >= 80, "Very High Discount")
    .when(col("discount_percentage") >= 50, "High Discount")
    .when(col("discount_percentage") >= 20, "Medium Discount")
    .otherwise("Low Discount")
)

In [ ]:
df.show(10)

In [ ]:
# column validation
from pyspark.sql.functions import col

def column_validation(df, column_name):
    null_count = df.filter(col(column_name).isNull()).count()
    if null_count == 0:
        print(f"Column: {column_name}, Nul= No")
    elif null_count > 1:
        print(f"Column: {column_name} Null = {null_count}")

for column in df.columns:
    column_validation(df, column)


In [ ]:
df.show(3)

In [ ]:
df.groupBy(["product_id"]).count().filter("count > 1").count()

In [ ]:
df.groupBy(["user_id"]).count().filter("count > 1").count()

In [ ]:
df.groupBy(["user_name"]).count().filter("count > 1").count()

In [ ]:
df.groupBy(["review_id"]).count().filter("count > 1").count()

In [ ]:
df.groupBy(["review_title"]).count().filter("count > 1").count()

In [ ]:
df.groupBy(["product_id", "user_id", "review_id"]).count().filter("count > 1").count()

In [ ]:
df.groupBy(["product_id", "user_id", "user_name"]).count().filter("count > 1").count()

In [ ]:
df.groupBy(["product_id", "user_id", "review_id", "rating", "discounted_price"]).count().filter("count > 1").count()

In [ ]:
# duplicate check
df.groupBy(df.columns).count().filter("count > 1").show(4)

In [ ]:
print(df.count())
df.show(2)

### save cleaned data

In [ ]:
from pathlib import Path

project_root = Path.cwd()
cleaned_data_dir = project_root / "data" / "processed"
cleaned_data_dir.mkdir(parents=True, exist_ok=True)

cleaned_data_path = cleaned_data_dir / "amazon_sales_cleaned"

print("Total cleaned data: ", df.count())
df.write \
    .mode("overwrite") \
    .parquet(str(cleaned_data_path))


#### The dataset is product centric 
This table is product-centric. Each row represents a product snapshot. Review and user fields are descriptive metadata and not unique identifiers
- as product id is not unique
- user_id/name or review_id/title is not atomic
- demormalized dataset

## Gold Layer

In [ ]:
from pathlib import Path

project_root = Path.cwd()
cleaned_data_dir = project_root / "data" / "processed"
cleaned_data_dir.mkdir(parents=True, exist_ok=True)

cleaned_data_path = cleaned_data_dir / "amazon_sales_cleaned"

cleaned_df = spark1.read.parquet(str(cleaned_data_path))

print(cleaned_df.count())
cleaned_df.show(20)

In [ ]:
# column validation
from functools import reduce
from pyspark.sql.functions import col

def column_validation(df, column_name):
    null_count = df.filter(col(column_name).isNull()).count()
    if null_count == 0:
        print(f"Column: {column_name}, Nul= No")
    elif null_count > 1:
        print(f"Column: {column_name} Null = {null_count}")

for column in cleaned_df.columns:
    column_validation(cleaned_df, column)


### products gold table

In [ ]:
# products gold table
from pyspark.sql.functions import col, avg, max, min, first, round as spark_round

products_gold_df = (
    cleaned_df
    .groupBy("product_id")
    .agg(
        first("product_title", ignorenulls=True).alias("product_title"),
        first("category", ignorenulls=True).alias("category"),
        spark_round(avg("rating"), 2).alias("avg_rating"),
        max("rating_count").alias("rating_count"),
        min("discounted_price").alias("discounted_price"),
        first("actual_price", ignorenulls=True).alias("actual_price"),
        first("discount_percentage", ignorenulls=True).alias("discount_percentage"),
        first("price_difference", ignorenulls=True).alias("price_difference"),
        first("rating_bucket", ignorenulls=True).alias("rating_bucket"),
        first("rating_strength", ignorenulls=True).alias("rating_strength")
    )
)

print(products_gold_df.count())
products_gold_df.show(10)

In [ ]:
# save into data folder
project_root = Path.cwd()
cleaned_data_dir = project_root / "data" / "gold"
cleaned_data_dir.mkdir(parents=True, exist_ok=True)

amazon_gold_data = cleaned_data_dir / "gold_products_summary"

products_gold_df.write.mode("overwrite").parquet(str(amazon_gold_data))


In [ ]:
from pyspark.sql.utils import AnalysisException
from py4j.protocol import Py4JJavaError
jdbc_url = "jdbc:postgresql://localhost:5432/airflow"

props = {
    "user": "airflow",
    "password": "airflow"
}
 
try:
    df.write \
      .mode("append") \
      .jdbc(
          url=jdbc_url,
          table="product_summary",
          properties=props
      )
    print("Data successfully written to product_summary")

except AnalysisException as e:
    print("ANALYSIS EXCEPTION")
    print(e)

except Py4JJavaError as e:
    print("JAVA ERROR (root cause below)")
    print(e.java_exception) 

except Exception as e:
    print("UNKNOWN ERROR")
    print(e)


In [ ]:
spark1._jvm.java.lang.Class.forName("org.postgresql.Driver")
print("PostgreSQL JDBC driver is available")


### category gold table

In [ ]:
cleaned_df.show(1)

In [ ]:
from pyspark.sql.functions import avg, count, round as spark_round

category_gold_df = cleaned_df.groupBy("category").agg(
    count("product_id").alias("total_products"),
    spark_round(avg("rating"), 2).alias("avg_rating"),
    spark_round(avg("rating_count"), 0).alias("avg_rating_count"),
    spark_round(avg("discount_percentage"), 2).alias("avg_discount_percentage"),
    spark_round(avg("price_difference"), 2).alias("avg_price_difference")
)

print(category_gold_df.count())
print(category_gold_df.printSchema())
category_gold_df.show(2)

In [ ]:
project_root = Path.cwd()
gold_data_dir = project_root / "data" / "gold"
gold_data_dir.mkdir(parents=True, exist_ok=True)
category_gold_data_path = gold_data_dir / "gold_category_summary"

category_gold_df.write.mode("overwrite").parquet(str(category_gold_data_path))


In [ ]:
category_gold_df.printSchema()

### Top products performance gold table

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import when, col, dense_rank

from pyspark.sql.functions import when, col

review_strength_score = when(col("rating_count") >= 10000, 4) \
    .when(col("rating_count") >= 1000, 3) \
    .when(col("rating_count") >= 100, 2) \
    .otherwise(1)

rating_bucket_score = when(col("rating") >= 4.5, 4) \
    .when(col("rating") >= 4.0, 3) \
    .when(col("rating") >= 3.0, 2) \
    .otherwise(1)

window_spec = Window.partitionBy("category").orderBy(
    col("rating_count").desc(),
    col("review_strength_score").desc(),
    col("rating_bucket_score").desc()
)


top_products_gold_df = (
    cleaned_df
    .withColumn("review_strength_score", review_strength_score)
    .withColumn("rating_bucket_score", rating_bucket_score)
    .withColumn("rank", dense_rank().over(window_spec))
    .filter(col("rank") <= 5)
    .select(
        "product_id",
        "product_title",
        "category",
        "rating",
        "rating_count",
        "discounted_price",
        "rank"
    )
)

print(top_products_gold_df.count())
top_products_gold_df.show(10)

In [ ]:
project_root = Path.cwd()
gold_data_dir = project_root / "data" / "gold"
gold_data_dir.mkdir(parents=True, exist_ok=True)
category_gold_data_path = gold_data_dir / "gold_top_products"

top_products_gold_df.write.mode("overwrite").parquet(str(category_gold_data_path))

In [ ]:
top_products_gold_df.printSchema()

### Discount gold tables

In [ ]:
from pyspark.sql.functions import col, avg, round as spark_round
discount_effectiveness_gold_df = cleaned_df.groupBy("discount_bucket") \
    .agg(
        spark_round(avg(col("rating")), 2).alias("avg_rating"),
        spark_round(avg(col("rating_count")), 2).alias("avg_rating_count"),
        count(col("product_id")).alias("total_products")
    )

print(discount_effectiveness_gold_df.count())
discount_effectiveness_gold_df.show(5)

In [ ]:
project_root = Path.cwd()
gold_data_dir = project_root / "data" / "gold"
gold_data_dir.mkdir(parents=True, exist_ok=True)
discount_effectiveness_gold_data_path = gold_data_dir / "gold_discount_effectiveness"

discount_effectiveness_gold_df.write.mode("overwrite").parquet(str(discount_effectiveness_gold_data_path))

In [ ]:
discount_effectiveness_gold_df.printSchema()